# PySpark + Microsoft SQL Server

This notebook connects PySpark to `CybersecurityDB`, reads `dbo.CybersecurityAttacks`, verifies the 40,000 records, and performs PySpark analysis.

In [16]:
import os
from dotenv import load_dotenv
from pyspark.sql import SparkSession, functions as F

load_dotenv()
print("Imports completed successfully.")


Imports completed successfully.


## 1. Create the Spark session

In [17]:
spark = (
    SparkSession.builder
    .appName("Cybersecurity_SQL_Server_Analysis")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        "com.microsoft.sqlserver:mssql-jdbc:12.8.1.jre11"
    )
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)
print("Spark session started successfully.")


Spark version: 4.1.1
Spark session started successfully.


## 2. Configure the SQL Server connection

In [18]:
db_server = "localhost"
db_port = "1433"
db_name = "CybersecurityDB"
db_user = "cyber_user"
db_password = "Cyber@2026Strong"

jdbc_url = (
    f"jdbc:sqlserver://{db_server}:{db_port};"
    f"databaseName={db_name};"
    "encrypt=true;"
    "trustServerCertificate=true;"
)

connection_properties = {
    "user": db_user,
    "password": db_password,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver",
}

print("JDBC URL:", jdbc_url)
print("Database:", db_name)
print("User:", db_user)

JDBC URL: jdbc:sqlserver://localhost:1433;databaseName=CybersecurityDB;encrypt=true;trustServerCertificate=true;
Database: CybersecurityDB
User: cyber_user


## 3. Read the SQL Server table with PySpark

In [19]:
table_name = "dbo.CybersecurityAttacks"

sql_df = spark.read.jdbc(
    url=jdbc_url,
    table=table_name,
    properties=connection_properties,
)

row_count = sql_df.count()
print(f"Successfully loaded {row_count:,} rows from SQL Server.")

if row_count != 40000:
    print("Warning: expected 40,000 rows, but found", row_count)

sql_df.printSchema()
sql_df.show(5, truncate=False)


Successfully loaded 40,000 rows from SQL Server.
root
 |-- Timestamp: timestamp (nullable = true)
 |-- Source_IP_Address: string (nullable = true)
 |-- Destination_IP_Address: string (nullable = true)
 |-- Source_Port: long (nullable = true)
 |-- Destination_Port: long (nullable = true)
 |-- Protocol: string (nullable = true)
 |-- Packet_Length: long (nullable = true)
 |-- Packet_Type: string (nullable = true)
 |-- Traffic_Type: string (nullable = true)
 |-- Payload_Data: string (nullable = true)
 |-- Malware_Indicators: string (nullable = true)
 |-- Anomaly_Scores: double (nullable = true)
 |-- Alerts_Warnings: string (nullable = true)
 |-- Attack_Type: string (nullable = true)
 |-- Attack_Signature: string (nullable = true)
 |-- Action_Taken: string (nullable = true)
 |-- Severity_Level: string (nullable = true)
 |-- User_Information: string (nullable = true)
 |-- Device_Information: string (nullable = true)
 |-- Network_Segment: string (nullable = true)
 |-- Geo_location_Data: strin

## 4. Data-quality checks

In [20]:
print("Number of columns:", len(sql_df.columns))
print("Column names:")
print(sql_df.columns)

null_summary = sql_df.select([
    F.sum(F.col(column).isNull().cast("int")).alias(column)
    for column in sql_df.columns
])

null_summary.show(truncate=False)


Number of columns: 25
Column names:
['Timestamp', 'Source_IP_Address', 'Destination_IP_Address', 'Source_Port', 'Destination_Port', 'Protocol', 'Packet_Length', 'Packet_Type', 'Traffic_Type', 'Payload_Data', 'Malware_Indicators', 'Anomaly_Scores', 'Alerts_Warnings', 'Attack_Type', 'Attack_Signature', 'Action_Taken', 'Severity_Level', 'User_Information', 'Device_Information', 'Network_Segment', 'Geo_location_Data', 'Proxy_Information', 'Firewall_Logs', 'IDS_IPS_Alerts', 'Log_Source']
+---------+-----------------+----------------------+-----------+----------------+--------+-------------+-----------+------------+------------+------------------+--------------+---------------+-----------+----------------+------------+--------------+----------------+------------------+---------------+-----------------+-----------------+-------------+--------------+----------+
|Timestamp|Source_IP_Address|Destination_IP_Address|Source_Port|Destination_Port|Protocol|Packet_Length|Packet_Type|Traffic_Type|Paylo

## 5. PySpark analysis

In [21]:
attack_type_analysis = (
    sql_df.groupBy("Attack_Type")
    .agg(F.count("*").alias("Total_Attacks"))
    .orderBy(F.desc("Total_Attacks"))
)

attack_type_analysis.show(truncate=False)


+-----------+-------------+
|Attack_Type|Total_Attacks|
+-----------+-------------+
|DDoS       |13428        |
|Malware    |13307        |
|Intrusion  |13265        |
+-----------+-------------+



In [22]:
severity_analysis = (
    sql_df.groupBy("Severity_Level")
    .agg(
        F.count("*").alias("Total_Incidents"),
        F.round(F.avg("Anomaly_Scores"), 2).alias("Average_Anomaly_Score"),
        F.round(F.avg("Packet_Length"), 2).alias("Average_Packet_Length"),
    )
    .orderBy(F.desc("Total_Incidents"))
)

severity_analysis.show(truncate=False)


+--------------+---------------+---------------------+---------------------+
|Severity_Level|Total_Incidents|Average_Anomaly_Score|Average_Packet_Length|
+--------------+---------------+---------------------+---------------------+
|Medium        |13435          |49.81                |783.58               |
|High          |13382          |50.3                 |780.83               |
|Low           |13183          |50.24                |779.92               |
+--------------+---------------+---------------------+---------------------+



In [23]:
yearly_analysis = (
    sql_df.withColumn("Year", F.year("Timestamp"))
    .groupBy("Year")
    .agg(F.count("*").alias("Total_Attacks"))
    .orderBy("Year")
)

yearly_analysis.show(truncate=False)


+----+-------------+
|Year|Total_Attacks|
+----+-------------+
|2020|10573        |
|2021|10538        |
|2022|10750        |
|2023|8139         |
+----+-------------+



In [24]:
protocol_analysis = (
    sql_df.groupBy("Protocol")
    .agg(
        F.count("*").alias("Total_Incidents"),
        F.round(F.avg("Anomaly_Scores"), 2).alias("Average_Anomaly_Score"),
    )
    .orderBy(F.desc("Total_Incidents"))
)

protocol_analysis.show(truncate=False)


+--------+---------------+---------------------+
|Protocol|Total_Incidents|Average_Anomaly_Score|
+--------+---------------+---------------------+
|ICMP    |13429          |50.12                |
|UDP     |13299          |49.99                |
|TCP     |13272          |50.23                |
+--------+---------------+---------------------+



In [25]:
attack_severity_analysis = (
    sql_df.groupBy("Attack_Type", "Severity_Level")
    .agg(F.count("*").alias("Incident_Count"))
    .orderBy(F.desc("Incident_Count"))
)

attack_severity_analysis.show(30, truncate=False)


+-----------+--------------+--------------+
|Attack_Type|Severity_Level|Incident_Count|
+-----------+--------------+--------------+
|DDoS       |High          |4523          |
|Malware    |Medium        |4516          |
|Intrusion  |Medium        |4464          |
|DDoS       |Medium        |4455          |
|DDoS       |Low           |4450          |
|Malware    |High          |4432          |
|Intrusion  |High          |4427          |
|Intrusion  |Low           |4374          |
|Malware    |Low           |4359          |
+-----------+--------------+--------------+



In [26]:
action_analysis = (
    sql_df.groupBy("Action_Taken")
    .agg(
        F.count("*").alias("Total_Actions"),
        F.round(F.avg("Anomaly_Scores"), 2).alias("Average_Anomaly_Score"),
    )
    .orderBy(F.desc("Total_Actions"))
)

action_analysis.show(truncate=False)


+------------+-------------+---------------------+
|Action_Taken|Total_Actions|Average_Anomaly_Score|
+------------+-------------+---------------------+
|Blocked     |13529        |50.14                |
|Ignored     |13276        |49.93                |
|Logged      |13195        |50.27                |
+------------+-------------+---------------------+



## 6. Convert only summarized results to Pandas

In [27]:
attack_type_pdf = attack_type_analysis.toPandas()
severity_pdf = severity_analysis.toPandas()
yearly_pdf = yearly_analysis.toPandas()
protocol_pdf = protocol_analysis.toPandas()
attack_severity_pdf = attack_severity_analysis.toPandas()
action_pdf = action_analysis.toPandas()

print("Aggregated results are ready for visualization.")
display(attack_type_pdf)


Aggregated results are ready for visualization.


,Attack_Type,Total_Attacks
0,DDoS,13428
1,Malware,13307
2,Intrusion,13265


## 7. Stop Spark

In [28]:
# spark.stop()
# print("Spark session stopped.")
